In [1]:
import jax
import jax.numpy as jnp
from jax import random, lax, grad, jit
import numpy as np

# --- RNN Configuration
input_dim = 1       # Each sample is a scalar
hidden_dim = 32     # Number of hidden units in the RNN cell
output_dim = 1      # We'll predict a scalar per sequence
seq_length = 100    # Length of each sequence
num_sequences = 10_000  # With 1M samples, we get 10k sequences

# --- RNN Parameter Initialization
def init_rnn_params(key, input_dim, hidden_dim, output_dim):
    k1, k2, k3, k4 = random.split(key, 4)
    W_xh = random.normal(k1, (input_dim, hidden_dim)) * 0.1
    W_hh = random.normal(k2, (hidden_dim, hidden_dim)) * 0.1
    b_h = jnp.zeros((hidden_dim,))
    W_hy = random.normal(k3, (hidden_dim, output_dim)) * 0.1
    b_y = jnp.zeros((output_dim,))
    return {"W_xh": W_xh, "W_hh": W_hh, "b_h": b_h, "W_hy": W_hy, "b_y": b_y}

# --- RNN Cell & Forward Pass
def rnn_step(params, h, x):
    # x: (input_dim,), h: (hidden_dim,)
    h_next = jnp.tanh(jnp.dot(x, params["W_xh"]) + jnp.dot(h, params["W_hh"]) + params["b_h"])
    return h_next

def rnn_forward(params, inputs):
    # inputs: (seq_length, input_dim)
    def step_fn(h, x):
        h_new = rnn_step(params, h, x)
        return h_new, h_new
    h0 = jnp.zeros((hidden_dim,))
    final_h, hidden_states = lax.scan(step_fn, h0, inputs)
    # Produce output from final hidden state.
    output = jnp.dot(final_h, params["W_hy"]) + params["b_y"]
    return output, hidden_states

# --- Loss Function (MSE)
def loss_fn(params, batch_inputs, batch_targets):
    # batch_inputs: (batch_size, seq_length, input_dim)
    # batch_targets: (batch_size, output_dim)
    def single_loss(inputs, target):
        pred, _ = rnn_forward(params, inputs)
        return jnp.mean((pred - target) ** 2)
    losses = jax.vmap(single_loss)(batch_inputs, batch_targets)
    return jnp.mean(losses)

# --- Training Step
@jit
def train_step(params, batch_inputs, batch_targets, learning_rate=0.001):
    grads = grad(loss_fn)(params, batch_inputs, batch_targets)
    new_params = {k: params[k] - learning_rate * grads[k] for k in params}
    loss = loss_fn(params, batch_inputs, batch_targets)
    return new_params, loss

# --- Dummy Data for Testing
# Assume processed_data is a 1D array of length 1M from our pipeline.
# For this example, we create dummy processed data.
processed_data = np.linspace(-1, 1, 1_000_000).astype(np.float32)
# Reshape into sequences: (num_sequences, seq_length, 1)
rnn_inputs = processed_data.reshape((num_sequences, seq_length, 1))
# Dummy targets: for example, the mean of the sequence (or any simple function)
rnn_targets = np.mean(rnn_inputs, axis=1)

# --- Training Loop
key = random.PRNGKey(0)
params = init_rnn_params(key, input_dim, hidden_dim, output_dim)
num_epochs = 5
batch_size = 32
num_batches = num_sequences // batch_size

for epoch in range(num_epochs):
    epoch_loss = 0.0
    for i in range(num_batches):
        batch_inputs = rnn_inputs[i*batch_size:(i+1)*batch_size]
        batch_targets = rnn_targets[i*batch_size:(i+1)*batch_size]
        params, loss_val = train_step(params, batch_inputs, batch_targets)
        epoch_loss += loss_val
    epoch_loss /= num_batches
    print(f"Epoch {epoch+1}, Loss: {epoch_loss:.6f}")

# --- Testing the RNN on a single sample
sample_input = rnn_inputs[0]
pred, _ = rnn_forward(params, sample_input)
print("Sample prediction:", pred)
print("Sample target:", rnn_targets[0])


Epoch 1, Loss: 0.250219
Epoch 2, Loss: 0.139215
Epoch 3, Loss: 0.047657
Epoch 4, Loss: 0.006999
Epoch 5, Loss: 0.000735
Sample prediction: [-0.9407144]
Sample target: [-0.99990106]


In [2]:
import jax
import jax.numpy as jnp
from jax import random, lax, grad, jit
import numpy as np

# --- RNN Configuration
input_dim = 1       # Each sample is a scalar
hidden_dim = 32     # Number of hidden units in the RNN cell
output_dim = 1      # We'll predict a scalar per sequence
seq_length = 100    # Length of each sequence
num_sequences = 10_000  # With 1M samples, we get 10k sequences

# --- RNN Parameter Initialization
def init_rnn_params(key, input_dim, hidden_dim, output_dim):
    k1, k2, k3, k4 = random.split(key, 4)
    W_xh = random.normal(k1, (input_dim, hidden_dim)) * 0.1
    W_hh = random.normal(k2, (hidden_dim, hidden_dim)) * 0.1
    b_h = jnp.zeros((hidden_dim,))
    W_hy = random.normal(k3, (hidden_dim, output_dim)) * 0.1
    b_y = jnp.zeros((output_dim,))
    return {"W_xh": W_xh, "W_hh": W_hh, "b_h": b_h, "W_hy": W_hy, "b_y": b_y}

# --- RNN Cell & Forward Pass
def rnn_step(params, h, x):
    # Compute next hidden state using tanh activation.
    h_next = jnp.tanh(jnp.dot(x, params["W_xh"]) + jnp.dot(h, params["W_hh"]) + params["b_h"])
    return h_next

def rnn_forward(params, inputs):
    # inputs: (seq_length, input_dim)
    def step_fn(h, x):
        h_new = rnn_step(params, h, x)
        return h_new, h_new
    h0 = jnp.zeros((hidden_dim,))
    final_h, hidden_states = lax.scan(step_fn, h0, inputs)
    # Produce output from final hidden state.
    output = jnp.dot(final_h, params["W_hy"]) + params["b_y"]
    return output, hidden_states

# --- Loss Function (MSE)
def loss_fn(params, batch_inputs, batch_targets):
    # batch_inputs: (batch_size, seq_length, input_dim)
    # batch_targets: (batch_size, output_dim)
    def single_loss(inputs, target):
        pred, _ = rnn_forward(params, inputs)
        return jnp.mean((pred - target) ** 2)
    losses = jax.vmap(single_loss)(batch_inputs, batch_targets)
    return jnp.mean(losses)

# --- Training Step (JIT-compiled)
@jit
def train_step(params, batch_inputs, batch_targets, learning_rate=0.001):
    grads = grad(loss_fn)(params, batch_inputs, batch_targets)
    new_params = {k: params[k] - learning_rate * grads[k] for k in params}
    loss = loss_fn(params, batch_inputs, batch_targets)
    return new_params, loss

# --- Dummy Data for Testing
# Create dummy processed data: 1M samples ranging from -1 to 1.
processed_data = np.linspace(-1, 1, 1_000_000).astype(np.float32)
# Reshape into sequences: (num_sequences, seq_length, 1)
rnn_inputs = processed_data.reshape((num_sequences, seq_length, 1))
# Dummy targets: for instance, the mean of each sequence.
rnn_targets = np.mean(rnn_inputs, axis=1)

# --- Training Loop
key = random.PRNGKey(0)
params = init_rnn_params(key, input_dim, hidden_dim, output_dim)
num_epochs = 5
batch_size = 32
num_batches = num_sequences // batch_size

for epoch in range(num_epochs):
    epoch_loss = 0.0
    for i in range(num_batches):
        batch_inputs = rnn_inputs[i*batch_size:(i+1)*batch_size]
        batch_targets = rnn_targets[i*batch_size:(i+1)*batch_size]
        params, loss_val = train_step(params, batch_inputs, batch_targets)
        epoch_loss += loss_val
    epoch_loss /= num_batches
    print(f"Epoch {epoch+1}, Loss: {epoch_loss:.6f}")

# --- Testing the RNN on a Single Sample
sample_input = rnn_inputs[0]
pred, _ = rnn_forward(params, sample_input)
print("Sample prediction:", pred)
print("Sample target:", rnn_targets[0])


Epoch 1, Loss: 0.250219
Epoch 2, Loss: 0.139215
Epoch 3, Loss: 0.047657
Epoch 4, Loss: 0.006999
Epoch 5, Loss: 0.000735
Sample prediction: [-0.9407144]
Sample target: [-0.99990106]


In [3]:
import os
import time
import numpy as np
import jax
import jax.numpy as jnp
from jax import lax, jit, vmap, pmap, grad, random
from jax.experimental import pjit
from jax.sharding import Mesh, PositionalSharding
from functools import partial

# ================================
# Pipeline Configuration (Small Scale)
# ================================
PIPE_MAX_RECURSION_DEPTH    = 100_000      # Maximum recursion depth for pipeline
PIPE_TOTAL_DEPTH            = 100_000      # Total recursion depth (fused into one iteration)
PIPE_OPTIMAL_DEPTH_STEP     = PIPE_TOTAL_DEPTH  # Full fusion in one go
PIPE_DIMENSIONAL_CONSTRAINT = 0.8
PIPE_BATCH_SIZE             = 1_000_000    # 1M samples for prototyping

# Use up to 8 devices (or the number available)
PIPE_NUM_DEVICES = min(8, jax.device_count())
PIPE_LOCAL_BATCH_SIZE = PIPE_BATCH_SIZE // PIPE_NUM_DEVICES

PIPE_VAL_CLAMP_LOW  = -100.0
PIPE_VAL_CLAMP_HIGH =  100.0

# Set up device mesh for pjit on TPU/Colab
devices = jax.devices()[:PIPE_NUM_DEVICES]
mesh = Mesh(devices, ("data",))
sharding = PositionalSharding(mesh.devices.flat)

# Data Lake Directory
DATA_LAKE_DIR = "datalake"
if not os.path.exists(DATA_LAKE_DIR):
    os.makedirs(DATA_LAKE_DIR)

# -------------------------------
# Pipeline Functions
# -------------------------------
@jit
def pipe_dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, PIPE_MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * PIPE_DIMENSIONAL_CONSTRAINT

@jit
def pipe_dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, PIPE_MAX_RECURSION_DEPTH)
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / ((scale_factor * 20) + 1)) * PIPE_DIMENSIONAL_CONSTRAINT

@jit
def pipe_stabilize_depth(depth):
    return depth / (1 + jnp.log1p(depth + 1))

@partial(jit, static_argnames=["depth"])
def pipe_dppu_with_dynamic_pi_phi(x, depth=PIPE_OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    depth = pipe_stabilize_depth(jnp.minimum(depth, PIPE_MAX_RECURSION_DEPTH))
    def body_fn(i, val):
        pi_dyn  = pipe_dynamic_pi(i, scale_factor)
        phi_dyn = pipe_dynamic_phi(i, scale_factor)
        scale   = jnp.log1p(i + 1) * scale_factor * PIPE_DIMENSIONAL_CONSTRAINT
        safe_val = jnp.clip(val, PIPE_VAL_CLAMP_LOW, PIPE_VAL_CLAMP_HIGH)
        new_val = jnp.sin(safe_val * scale * pi_dyn) * jnp.exp(-safe_val / (phi_dyn + 10))
        return new_val
    return lax.fori_loop(0, depth.astype(jnp.int32), body_fn, x)

def pipe_branch_recycle(x, num_branches=2, branch_depth=PIPE_OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    xs = jnp.stack([x] * num_branches, axis=0)
    branch_fn = vmap(lambda xi: pipe_dppu_with_dynamic_pi_phi(xi, depth=branch_depth, scale_factor=scale_factor))
    branch_outputs = branch_fn(xs)
    return jnp.sum(branch_outputs, axis=0)

@partial(pjit.pjit,
         in_shardings=(sharding,),
         out_shardings=sharding,
         static_argnames=("num_branches", "branch_depth", "scale_factor"))
def pipe_process_full(x, num_branches, branch_depth, scale_factor):
    return pipe_branch_recycle(x, num_branches=num_branches, branch_depth=branch_depth, scale_factor=scale_factor)

def run_pipeline(batch_input, total_depth=PIPE_TOTAL_DEPTH, num_branches=2,
                 branch_depth=PIPE_OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    sharded_input = jax.device_put(batch_input, sharding)
    start_time = time.time()
    final_output = pipe_process_full(sharded_input, num_branches, branch_depth, scale_factor)
    final_output = jax.device_get(final_output)
    jax.block_until_ready(final_output)
    elapsed = time.time() - start_time
    mean_val = float(jnp.mean(final_output))
    return final_output, elapsed, mean_val

def save_to_data_lake(data, filename="processed_data.npy"):
    filepath = os.path.join(DATA_LAKE_DIR, filename)
    np.save(filepath, np.array(data))
    print(f"Saved processed data to {filepath}")

def load_from_data_lake(filename="processed_data.npy"):
    filepath = os.path.join(DATA_LAKE_DIR, filename)
    if os.path.exists(filepath):
        data = np.load(filepath)
        print(f"Loaded processed data from {filepath}")
        return data
    else:
        print(f"No file found at {filepath}")
        return None

# ================================
# RNN Configuration and Functions
# ================================
input_dim = 1
hidden_dim = 32
output_dim = 1
seq_length = 100  # Each sequence will be 100 samples

def get_num_sequences(data_length, seq_length):
    return data_length // seq_length

def init_rnn_params(key, input_dim, hidden_dim, output_dim):
    k1, k2, k3, k4 = random.split(key, 4)
    W_xh = random.normal(k1, (input_dim, hidden_dim)) * 0.1
    W_hh = random.normal(k2, (hidden_dim, hidden_dim)) * 0.1
    b_h = jnp.zeros((hidden_dim,))
    W_hy = random.normal(k3, (hidden_dim, output_dim)) * 0.1
    b_y = jnp.zeros((output_dim,))
    return {"W_xh": W_xh, "W_hh": W_hh, "b_h": b_h, "W_hy": W_hy, "b_y": b_y}

def rnn_step(params, h, x):
    h_next = jnp.tanh(jnp.dot(x, params["W_xh"]) + jnp.dot(h, params["W_hh"]) + params["b_h"])
    return h_next

def rnn_forward(params, inputs):
    def step_fn(h, x):
        h_new = rnn_step(params, h, x)
        return h_new, h_new
    h0 = jnp.zeros((hidden_dim,))
    final_h, _ = lax.scan(step_fn, h0, inputs)
    output = jnp.dot(final_h, params["W_hy"]) + params["b_y"]
    return output

def rnn_loss_fn(params, batch_inputs, batch_targets):
    def single_loss(inputs, target):
        pred = rnn_forward(params, inputs)
        return jnp.mean((pred - target) ** 2)
    losses = jax.vmap(single_loss)(batch_inputs, batch_targets)
    return jnp.mean(losses)

@jit
def rnn_train_step(params, batch_inputs, batch_targets, learning_rate=0.001):
    grads = grad(rnn_loss_fn)(params, batch_inputs, batch_targets)
    new_params = {k: params[k] - learning_rate * grads[k] for k in params}
    loss = rnn_loss_fn(params, batch_inputs, batch_targets)
    return new_params, loss

# ================================
# Main: End-to-End System Prototype
# ================================
if __name__ == "__main__":
    # ----- Pipeline Stage -----
    print("Running pipeline to process raw data...")
    raw_data = jnp.linspace(0, 10, PIPE_BATCH_SIZE)  # 1M samples from 0 to 10
    processed_data, pipe_elapsed, pipe_mean = run_pipeline(raw_data)
    print(f"Pipeline completed in {pipe_elapsed:.2f} sec with mean output {pipe_mean:.6f}")
    save_to_data_lake(processed_data, filename="processed_data.npy")

    # ----- Prepare Data for RNN -----
    # Our processed_data is a 1D array of length PIPE_BATCH_SIZE.
    processed_data_np = np.array(processed_data)
    total_length = processed_data_np.shape[0]
    num_sequences = get_num_sequences(total_length, seq_length)
    # Reshape into (num_sequences, seq_length, 1)
    rnn_inputs = processed_data_np[:num_sequences * seq_length].reshape((num_sequences, seq_length, 1))
    # For demonstration, use the mean of each sequence as the dummy target.
    rnn_targets = np.mean(rnn_inputs, axis=1)  # shape: (num_sequences, 1)
    print(f"RNN training data: {num_sequences} sequences of length {seq_length}")

    # ----- RNN Training Stage -----
    key = random.PRNGKey(0)
    rnn_params = init_rnn_params(key, input_dim, hidden_dim, output_dim)
    num_epochs = 5
    batch_size = 32
    num_batches = num_sequences // batch_size

    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for i in range(num_batches):
            batch_inputs = rnn_inputs[i*batch_size:(i+1)*batch_size]
            batch_targets = rnn_targets[i*batch_size:(i+1)*batch_size]
            rnn_params, loss_val = rnn_train_step(rnn_params, batch_inputs, batch_targets)
            epoch_loss += loss_val
        epoch_loss /= num_batches
        print(f"Epoch {epoch+1}, Loss: {epoch_loss:.6f}")

    # ----- RNN Testing Stage -----
    sample_input = rnn_inputs[0]
    sample_pred = rnn_forward(rnn_params, sample_input)
    print("Sample RNN prediction:", sample_pred)
    print("Sample RNN target:", rnn_targets[0])


Running pipeline to process raw data...
Pipeline completed in 351.71 sec with mean output -0.031019
Saved processed data to datalake/processed_data.npy
RNN training data: 10000 sequences of length 100
Epoch 1, Loss: 0.034772
Epoch 2, Loss: 0.033611
Epoch 3, Loss: 0.033175
Epoch 4, Loss: 0.032941
Epoch 5, Loss: 0.032787
Sample RNN prediction: [-0.06436943]
Sample RNN target: [0.1932038]
